# Export Data for Number_of_plots.jpg
This notebook processes the raw data and saves it to `../figure_data/figure_S_number_of_plots/` for use in the figure-making notebook.

In [1]:
import os

import pandas as pd
import xarray as xr

from conus_biomass import dir_info
from conus_biomass.make_figures import maps
from conus_biomass.process_inputs import bin_nfi_plots
from conus_biomass.train_models import train_all_models
from conus_biomass.train_models.train_models_utils import calculate_change_in_var

# Output directory
OUT_DIR = "../figure_data/figure_S_number_of_plots/"
os.makedirs(OUT_DIR, exist_ok=True)

## Load and filter FIA data

In [2]:
slope = xr.open_zarr(dir_info.dir_processed + "data_on_ref_grid/1000m/aspect.zarr")
target_crs = slope["spatial_ref"].crs_wkt

fia_data = train_all_models.load_data(
    fpath=dir_info.dir_processed + "restructured_FIA/" + "*_FIA_plots_and_PRISM_v7.nc"
)
fia_data = fia_data.where(
    fia_data["STATECD"].isin([4, 6, 8, 16, 30, 32, 35, 41, 49, 53, 56]).load(), drop=True
)
fia_data = fia_data.where(
    (~fia_data["biomass"].isnull().load()).where(fia_data["year"] > 1999).sum(dim="year") > 1,
    drop=True,
)

In [3]:
print((fia_data["fire_between_measurements"] == 0).sum().values)
print((fia_data["fire_between_measurements"] > 0).sum().values)

36603
2912


## Load modeled biomass and compute deltas

In [4]:
extracted_points = xr.open_dataset("../modeled_biomass_all_years_fia_points_buffer.nc")
extracted_points["plotid"] = extracted_points["plotid"].values.astype("U12")

fia_data["biomass_modeled"] = extracted_points["predicted_biomass"].transpose("plotid", "year")

modeled_delta = calculate_change_in_var(fia_data=fia_data, var="biomass_modeled").where(
    fia_data["measyear_2"] <= 2019
)
actual_delta = calculate_change_in_var(fia_data=fia_data, var="biomass").where(
    fia_data["measyear_2"] <= 2019
)

fia_data["biomass_delta_modeled"] = modeled_delta
fia_data["biomass_delta_actual"] = actual_delta

## Compute binned counts (modeled)

In [5]:
vars_year = ["biomass"]
vars_static_modeled = ["biomass_delta_modeled", "lat", "lon"]

ds_modeled = fia_data[vars_year + vars_static_modeled]

ds_binned_modeled_counts = bin_nfi_plots.calculate_ds_binned(
    ds=ds_modeled[["biomass_delta_modeled", "lat", "lon"]],
    vars_year=vars_year,
    vars_static=vars_static_modeled,
    do_counts=True,
)
ds_binned_modeled_counts["lat_bin"] = ds_binned_modeled_counts["lat_bin"] + (0.5 / 2)
ds_binned_modeled_counts["lon_bin"] = ds_binned_modeled_counts["lon_bin"] + (0.5 / 2)

## Reproject binned counts

In [6]:
counts_modeled_reproj = (
    ds_binned_modeled_counts["n_plots"]
    .rio.set_spatial_dims(x_dim="lon_bin", y_dim="lat_bin")
    .rio.write_crs("EPSG:4326")
    .rio.reproject(target_crs)
)

## Save outputs

In [8]:
# Save counts_modeled_reproj as a NetCDF
counts_modeled_reproj.to_dataset(name="n_plots").to_netcdf(OUT_DIR + "counts_modeled_reproj.nc")
print("Saved counts_modeled_reproj.nc")

# Save FIA plot lon/lat as CSV (used for scatter overlay)
fia_lonlat = pd.DataFrame(
    {
        "lon": fia_data["lon"].values,
        "lat": fia_data["lat"].values,
    }
)
fia_lonlat.to_csv(OUT_DIR + "fia_plot_locations.csv", index=False)
print("Saved fia_plot_locations.csv")

# Save target CRS string
with open(OUT_DIR + "target_crs.txt", "w") as f:
    f.write(target_crs)
print("Saved target_crs.txt")

# Save western shapefile
shp_native = maps.SHP_WESTERN.to_crs(target_crs)
shp_native.to_file(OUT_DIR + "shp_western_native.gpkg", driver="GPKG")
print("Saved shp_western_native.gpkg")

Saved counts_modeled_reproj.nc
Saved fia_plot_locations.csv
Saved target_crs.txt


INFO:pyogrio._io:Created 11 records


Saved shp_western_native.gpkg
